# FedSwarm — Phase 2.1 centralized ceiling (Kaggle GPU)

Trains the centralized performance ceiling every FL method in later phases gets measured against: `SimpleCNN` (primary config, 112px) across 5 seeds x {groupnorm, batchnorm}, plus an optional secondary `ResNet-18` @224 table.

**Before running:**
1. **Turn on GPU.** Notebook settings (right sidebar) -> Accelerator -> GPU T4 x2 (or P100). This does nothing on CPU except run very slowly.
2. **Add the dataset.** Right sidebar -> Add Input -> search **"Brain Tumor MRI Dataset"** by **masoudnickparvar** (slug `masoudnickparvar/brain-tumor-mri-dataset`) -> Add. This must be the *same* dataset used to build `manifest.csv` locally, or the manifest's file paths won't resolve against what's mounted here.
3. **Turn on internet** (needed to `git clone` and `pip install`). Settings -> Internet -> On.

This notebook does **not** need `flwr` — that's only required once the Flower client/server harness exists (Phase 3+). Centralized training is plain PyTorch.

In [ ]:
import subprocess

REPO_URL = "https://github.com/researchpaper784-alt/ResearchPaper.git"
REPO_DIR = "/kaggle/working/ResearchPaper"

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

In [ ]:
%cd /kaggle/working/ResearchPaper

# Kaggle's base image already has torch/torchvision/numpy/pandas/scikit-learn/
# scipy/matplotlib/pillow/tqdm. Only these are missing for centralized training
# (flwr/flwr-datasets/imagehash/kaggle are not imported by this training path).
!pip install -q omegaconf rich

In [ ]:
import glob
import os

# Auto-detect wherever Kaggle mounted the dataset -- fedswarm.data.download.find_split_parent
# searches recursively for Training/ and Testing/ under this root, so it doesn't need to be
# the exact leaf directory.
candidates = glob.glob("/kaggle/input/*")
assert candidates, "No dataset found under /kaggle/input -- did you add it in the sidebar?"
print("Found /kaggle/input entries:", candidates)

os.environ["FEDSWARM_DATA_ROOT"] = candidates[0]
print("FEDSWARM_DATA_ROOT =", os.environ["FEDSWARM_DATA_ROOT"])

In [ ]:
import sys
sys.path.insert(0, "src")

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected -- check Settings > Accelerator before continuing")

# Imported here, not at the top: it only resolves after the sys.path insert above.
from fedswarm.data.download import find_split_parent, resolve_root  # noqa: E402
split_parent = find_split_parent(resolve_root(None))
print("Resolved dataset split parent:", split_parent)

## Build the decoded-image cache

Uses the manifest already committed to the repo (`data/processed/manifest.csv`) -- the pseudo-patient-level split from Phase 1.3, with cross-split leakage verified zero. This step is I/O-bound (JPEG decode via PIL), not GPU-bound, and should take well under a minute for 7,200 images.

In [ ]:
!python -m fedswarm.data.cache --size 112

## Primary ceiling: SimpleCNN @112, 5 seeds x {groupnorm, batchnorm}

GroupNorm is the FL-relevant default; BatchNorm is run alongside as the A9 confound-control ablation (BatchNorm running statistics aggregate badly under non-IID federated data -- this comparison shows whether that's actually visible here, not just asserted). Progress prints after every epoch since output isn't buffered through a pipe here.

In [ ]:
import subprocess

for norm in ["groupnorm", "batchnorm"]:
    for seed in [0, 1, 2, 3, 4]:
        print(f"\n{'='*20} norm={norm} seed={seed} {'='*20}")
        subprocess.run([
            "python", "scripts/run_experiment.py",
            "--config", "configs/base.yaml", "configs/model/simple_cnn.yaml", "configs/experiment/centralized.yaml",
            "--seed", str(seed),
            "--override", f"model.norm={norm}",
        ], check=True)

In [ ]:
!python scripts/summarize_centralized.py

## Optional: secondary ResNet-18 @224 table

"Does the ceiling hold with a larger, pretrained backbone." More expensive than the primary sweep (224px + 11.2M params), so run this only if GPU quota allows -- skip this cell entirely to save quota if you just need the primary ceiling.

In [ ]:
!python -m fedswarm.data.cache --size 224

for seed in [0, 1, 2, 3, 4]:
    print(f"\n{'='*20} resnet18 seed={seed} {'='*20}")
    subprocess.run([
        "python", "scripts/run_experiment.py",
        "--config", "configs/base.yaml", "configs/model/resnet18.yaml", "configs/experiment/centralized_resnet18.yaml",
        "--seed", str(seed),
    ], check=True)

!python scripts/summarize_centralized.py

## Getting results back

`results/centralized/*.json` is what we need locally. Simplest path: this cell zips them into `/kaggle/working/` -- Kaggle preserves everything under `/kaggle/working` as notebook output, downloadable from the Output tab after a "Save & Run All" (or right now, from the file browser). Download the zip and hand it back.

In [ ]:
!cd results && zip -r /kaggle/working/centralized_results.zip centralized/
print("Wrote /kaggle/working/centralized_results.zip -- download it from the Output tab.")